# Training on Google Colab (free GPU)

This notebook runs the project's **command-line interface** on a Colab GPU. All the code lives in the
GitHub repo (`src/`); the notebook only clones it and calls `python -m src.cli ...`.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`.

What survives when Colab disconnects: everything written to **Google Drive** (checkpoints, curves,
MLflow database). The cells below put `outputs/` on Drive automatically.

| Run | Config | Approx. time on a T4 |
|---|---|---|
| Smoke test | `cls_scratch.yaml` + limits | ~2 min |
| From-scratch CNN | `cls_scratch.yaml` (100 epochs, early stopping) | ~45–70 min |
| ResNet-50 linear probe | `cls_resnet50_probe.yaml` (30 epochs) | ~15–20 min |
| ResNet-50 fine-tuning | `cls_resnet50.yaml` (30 epochs) | ~30–40 min |
| Grid search / Random search | `tune_grid.yaml` / `tune_random.yaml` (8 trials x 5 epochs each) | ~25–35 min each |

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("CUDA available:", torch.cuda.is_available())

## 2. Connect Google Drive (to keep results)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/DeepLearning-Project'   # change if you want another folder
!mkdir -p "$DRIVE_DIR/outputs"

## 3. Get the code (clone the first time, pull the latest version after that)

In [ ]:
REPO = 'https://github.com/Moazdarwishh/DeepLearning-Project.git'
%cd /content
!test -d DeepLearning-Project || git clone -q $REPO
%cd /content/DeepLearning-Project
!git pull -q
!git log --oneline -3

## 4. Install the few packages Colab does not already have
(PyTorch, torchvision, scikit-learn, matplotlib, pandas and OpenCV are pre-installed on Colab.)

In [ ]:
!pip install -q datasets mlflow imagehash pyspellchecker

## 5. Store `outputs/` on Drive and check the setup
`outputs/` becomes a link to your Drive folder, so checkpoints are saved there directly.
The MLflow database from earlier sessions is copied back, so new runs are added to it.

In [ ]:
!rm -rf outputs && ln -s "$DRIVE_DIR/outputs" outputs
!test -f "$DRIVE_DIR/mlflow.db" && cp "$DRIVE_DIR/mlflow.db" mlflow.db && echo "restored mlflow.db from Drive"
!ls -la outputs
!python -m src.cli info --config configs/cls_resnet50.yaml | head -6
!python -m pytest -q

## 6. Smoke test (≈2 min) — run this first every new session
Two tiny epochs. Accuracy will be ~0.5–1 % (random guessing over 200 classes); this only checks that
everything works on the GPU. The first run downloads the dataset (~1.2 GB, 1–2 min).

In [ ]:
!python -m src.cli train --task classification --config configs/cls_scratch.yaml \
    --set train.epochs=2 train.limit_train_batches=20 train.limit_val_batches=5 \
          tracking.backend=none experiment.name=smoke-colab

## 7. Full training runs
Run one cell at a time. Each run writes to `outputs/runs/<name>/` on Drive: `best.pt`, `last.pt`,
`history.json`, `curves.png`, `summary.json`, `config.yaml`. Keep the browser tab open (Colab stops
idle sessions after ~90 min).

**7a. From-scratch CNN (baseline)**

In [ ]:
!python -m src.cli train --task classification --config configs/cls_scratch.yaml

**7b. ResNet-50 linear probe (backbone frozen)**

In [ ]:
!python -m src.cli train --task classification --config configs/cls_resnet50_probe.yaml

**7c. ResNet-50 full fine-tuning**

In [ ]:
!python -m src.cli train --task classification --config configs/cls_resnet50.yaml

**7d. Hyperparameter tuning: Grid Search vs Random Search** (same budget: 8 trials x 5 epochs each,
ResNet-50 linear probe, ~25-35 min per search). If Colab disconnects, re-run the same cell: finished
trials are skipped automatically.

In [ ]:
!python -m src.cli tune --task classification --config configs/tune_grid.yaml

In [ ]:
!python -m src.cli tune --task classification --config configs/tune_random.yaml

**7e. Compare the two searches** (table + `outputs/tuning/comparison.png`)

In [ ]:
!python -m src.cli tune --task classification --compare outputs/tuning/tune-grid outputs/tuning/tune-random
from IPython.display import Image, display
display(Image('outputs/tuning/comparison.png'))
for s in ['tune-grid', 'tune-random']:
    display(Image(f'outputs/tuning/{s}/search.png'))

### If Colab disconnected in the middle of a run
Re-run sections 1–5, then resume from the last saved epoch (change the run name):

In [ ]:
RUN = 'cls-scratch'      # folder name inside outputs/runs/
!python -m src.cli train --task classification --config outputs/runs/$RUN/config.yaml \
    --resume outputs/runs/$RUN/last.pt

## 8. Compare the runs

In [ ]:
import json, glob, pandas as pd
rows = []
for f in sorted(glob.glob('outputs/runs/*/summary.json')):
    s = json.load(open(f)); v = s['best_val']
    rows.append({'run': s['run'], 'best_epoch': s['best_epoch'], 'epochs_run': s['epochs_run'],
                 'minutes': s['train_time_min'], 'trainable_params': s['params']['trainable'],
                 'val_acc': v['val/accuracy'], 'val_top5': v['val/top5_accuracy'], 'val_macro_f1': v['val/macro_f1']})
pd.DataFrame(rows).round(4)

In [ ]:
from IPython.display import Image, display
for f in sorted(glob.glob('outputs/runs/cls-*/curves.png')):
    print(f); display(Image(f))

## 9. Final evaluation on the TEST set (only once, after choosing the models)
The test set is never used during training or tuning — only here.

In [ ]:
for run in ['cls-scratch', 'cls-resnet50-probe', 'cls-resnet50-ft']:
    print('==', run)
    !python -m src.cli evaluate --task classification --checkpoint outputs/runs/$run/best.pt --split test

## 10. Save the MLflow database to Drive (run this at the end of every session)
To view all runs as a dashboard on your laptop: download `mlflow.db` from the Drive folder into the
project folder, then run `mlflow ui --backend-store-uri sqlite:///mlflow.db` and open http://127.0.0.1:5000.
(If you already have a local `mlflow.db`, save this one under another name and point `mlflow ui` to it.)

In [ ]:
!cp mlflow.db "$DRIVE_DIR/mlflow.db" && ls -la "$DRIVE_DIR"